In [1]:
# Library imports

import numpy as np
import pandas as pd

RANDOM_STATE = 42

In [2]:
# Z-Score Normalization Function

def compute_z_score_normalization(X_train):
    """
    Compute mean and variance from training data only. 
    
    Parameters
    ----------
    X_train : np.ndarray, shape (n_samples, n_features)
    
    Returns
    -------
    mu: np.array, shape (n_features,) - mean per feature
    sigma: np.array, shape (n_features,) - standard deviation per feature    
    """
    mu = np.mean(X_train, axis=0)
    sigma = np.std(X_train, axis=0, ddof=0)
    return mu, sigma
    
def apply_zscore(X, mu, sigma):
    """
    Apply z-score standardization using pre-computed params.
    z(x) = (x - mu) / sigma
    
    Parameters
    ----------
    X     : np.ndarray  — data to normalize (train or test)
    mu    : np.ndarray  — mean from training set
    sigma : np.ndarray  — std dev from training set
    
    Returns
    -------
    np.ndarray — normalized data, same shape as X
    """
    return (X - mu) / sigma 

In [3]:
# Breast Cancer Wisconsin Dataset Loader

def load_breast_cancer(filepath):
    cols = ["sample_code_number", "clump_thickness", "uniformity_of_cell_size",
            "uniformity_of_cell_shape", "marginal_adhesion", "single_epithelial_cell_size",
            "bare_nuclei", "bland_chromatin", "normal_nucleoli", "mitoses", "class"]
    
    df = pd.read_csv(filepath, header=None, names=cols, na_values='?')
    
    # drop ID - not a feature
    df.drop(["sample_code_number"], axis=1, inplace=True)
    
    # bare_nuclei is discrete (integer 1-10) impute with MODE following requirements
    mode_val = df["bare_nuclei"].mode()[0]
    df["bare_nuclei"] = df["bare_nuclei"].fillna(mode_val)
    
    # encode label: 2 = benign (0), 4 = malignant (1)
    df["class"] = df["class"].map({2: 0, 4: 1})
    
    # all features are discrete integers 1-10, convert to float for normalization
    X = df.drop(columns=["class"]).values.astype(float)
    Y = df["class"].values.astype(int)
    
    return X, Y

bc_X, bc_Y = load_breast_cancer("../Data Sets/breast-cancer-wisconsin.data")

# Smoke test
assert bc_X.shape == (699, 9), f"Unexpected shape: {bc_X.shape}"
assert not np.isnan(bc_X).any(), "NaNs remain in breast cancer X"
assert set(np.unique(bc_Y)) == {0, 1}
print("Breast Cancer OK:", bc_X.shape, bc_Y.shape)

Breast Cancer OK: (699, 9) (699,)


In [4]:
# Congressional Voting Records Dataset Loader

def load_congressional_voting(filepath):
    cols = ["class", "handicapped_infants", "water_project_cost_sharing", "adoption_of_the_budget_resolution",
            "physician_fee_freeze", "el_salvador_aid", "religious_groups_in_schools", "anti_satellite_test_ban", "aid_to_nicaraguan_contras",
            "mx_missile", "immigration", "synfuels_corporation_cutback", "education_spending", "superfund_right_to_sue", "crime", "duty_free_exports",
            "export_administration_act_south_africa"]
    
    df = pd.read_csv(filepath, header=None, names=cols)
    
    # ? = abstain, valid third category, not missing. 
    # integer encode: y=1, n=0, ?=2
    # These are nominal categorical - distance handling will be done in disctance function, not here. CHANGE HERE IF GOING TO USE VDM
    vote_map = {"y": 1, "n": 0, "?": 2}
    for col in cols[1:]: 
        df[col] = df[col].map(vote_map)
        
    # Encode Label
    df["class"] = df["class"].map({"democrat": 0, "republican": 1})
    
    X = df.drop(columns=["class"]).values.astype(int)
    Y = df["class"].values.astype(int)
    
    return X, Y

vote_X, vote_Y = load_congressional_voting("../Data Sets/house-votes-84.data")

# Smoke test
assert vote_X.shape == (435, 16)
assert not np.isnan(vote_X).any()
assert set(np.unique(vote_Y)) == {0, 1}
print("Congressional Vote OK:", vote_X.shape, vote_Y.shape)


Congressional Vote OK: (435, 16) (435,)


In [5]:
# Computer Hardware Dataset Loader

def load_computer_hardware(filepath):
    cols = ["vendor_name", "model_name", "myct", "mmin", "mmax", "cache", "chmin", "chmax", "prp", "erp"]
    
    df = pd.read_csv(filepath, header=None, names=cols)
    
    # drop non-featuers per requirements
    # Save ERP separately, requirements says it might be useful for later. 
    
    erp = df["erp"].values.copy()
    df.drop(columns=["vendor_name", "model_name", "erp"], inplace=True)
    
    assert df.isnull().sum().sum() == 0, "Unexpected NaNs in computer hardware dataset"    
    
    X = df.drop(columns=["prp"]).values.astype(float)
    Y = df["prp"].values.astype(float)
    
    return X, Y, erp

hw_X, hw_y, hw_erp = load_computer_hardware("../Data Sets/machine.data")

# Smoke test
assert hw_X.shape == (209, 6)
assert not np.isnan(hw_X).any()
print("Computer Hardware OK:", hw_X.shape, hw_y.shape)
print("ERP saved for comparison:", hw_erp.shape)


Computer Hardware OK: (209, 6) (209,)
ERP saved for comparison: (209,)


In [6]:
# Abalone Dataset Loader

def load_abalone(filepath):
    cols = [
        "sex", "length", "diameter", "height",
        "whole_weight", "shucked_weight", "viscera_weight",
        "shell_weight", "rings"
    ]
    df = pd.read_csv(filepath, header=None, names=cols)

    # Sex is nominal — one-hot encode (M, F, I have no ordinal relationship)
    df = pd.get_dummies(df, columns=["sex"], drop_first=False)
    # Produces: sex_F, sex_I, sex_M

    assert df.isnull().sum().sum() == 0, "Unexpected NaNs in abalone"

    X = df.drop(columns=["rings"]).values.astype(float)
    y = df["rings"].values.astype(float)

    return X, y


ab_X, ab_y = load_abalone("../Data Sets/abalone.data")

assert ab_X.shape == (4177, 10)
assert not np.isnan(ab_X).any()
print("Abalone OK:", ab_X.shape, ab_y.shape)


Abalone OK: (4177, 10) (4177,)


In [7]:
# Bundle all datasets into a dictionary for easy access in later steps

datasets = {"breast_cancer":  {"X": bc_X,   "y": bc_Y,   "task": "classification"},
            "congress_vote":  {"X": vote_X, "y": vote_Y, "task": "classification"},
            "hw_performance": {"X": hw_X,   "y": hw_y,   "task": "regression",
                       "erp": hw_erp},
            "abalone":        {"X": ab_X,   "y": ab_y,   "task": "regression"},
}

for name, d in datasets.items():
    print(f"{name:20s} | {d['task']:14s} | X: {str(d['X'].shape):12s} | y: {d['y'].shape}")


breast_cancer        | classification | X: (699, 9)     | y: (699,)
congress_vote        | classification | X: (435, 16)    | y: (435,)
hw_performance       | regression     | X: (209, 6)     | y: (209,)
abalone              | regression     | X: (4177, 10)   | y: (4177,)


In [8]:
# Tuning Split Function

def make_tuning_split(X, y, task, tuning_frac=0.20, random_state=42):
    """
    Separate a held-out tuning set (20%) from the main data (80%).
    
    For classification: stratified so class proportions are preserved.
    For regression: uniform sampling across the sorted target range.
    
    Parameters
    ----------
    X            : np.ndarray, shape (n, d)
    y            : np.ndarray, shape (n,)
    task         : str — 'classification' or 'regression'
    tuning_frac  : float — fraction to hold out for tuning
    random_state : int
    
    Returns
    -------
    X_main, y_main   : 80% used for CV
    X_tune, y_tune   : 20% held out for hyperparameter tuning
    """
    rng = np.random.default_rng(random_state)
    n = len(y)
    tune_idx = []

    if task == 'classification':
        # Stratified: sample tuning_frac from each class independently
        classes = np.unique(y)
        for c in classes:
            class_idx = np.where(y == c)[0]
            n_tune = max(1, int(np.round(len(class_idx) * tuning_frac)))
            chosen = rng.choice(class_idx, size=n_tune, replace=False)
            tune_idx.extend(chosen.tolist())

    elif task == 'regression':
        # Uniform: sort by target, take every 1/tuning_frac-th point
        sorted_idx = np.argsort(y)
        step = int(round(1.0 / tuning_frac))
        tune_idx = sorted_idx[::step].tolist()
        # Trim if we overshot 20%
        n_tune_target = int(np.round(n * tuning_frac))
        tune_idx = tune_idx[:n_tune_target]

    tune_idx = np.array(tune_idx)
    main_idx = np.setdiff1d(np.arange(n), tune_idx)

    return X[main_idx], y[main_idx], X[tune_idx], y[tune_idx]

In [9]:
# Apply tuning split to all datasets and store results back in the dictionary

for name, d in datasets.items():
    X_main, y_main, X_tune, y_tune = make_tuning_split(
        d["X"], d["y"], d["task"], random_state=RANDOM_STATE
    )
    d["X_main"]  = X_main
    d["y_main"]  = y_main
    d["X_tune"]  = X_tune
    d["y_tune"]  = y_tune

# Verify sizes
for name, d in datasets.items():
    n_total = len(d["y"])
    n_main  = len(d["y_main"])
    n_tune  = len(d["y_tune"])
    pct     = n_tune / n_total * 100
    print(f"{name:20s} | total: {n_total:5d} | main: {n_main:5d} | tune: {n_tune:4d} ({pct:.1f}%)")

breast_cancer        | total:   699 | main:   559 | tune:  140 (20.0%)
congress_vote        | total:   435 | main:   348 | tune:   87 (20.0%)
hw_performance       | total:   209 | main:   167 | tune:   42 (20.1%)
abalone              | total:  4177 | main:  3342 | tune:  835 (20.0%)


In [10]:
# 5x2 CV Fold Generator

def make_5x2_folds(X, y, task, random_state=42):
    """
    Generate 5 rounds of 2-fold CV splits from the main 80% data.
    
    Each round: randomly split data in half.
      - Classification: stratified split
      - Regression: uniform split (sort by target, interleave)
    
    Parameters
    ----------
    X    : np.ndarray — main data (already has tuning set removed)
    y    : np.ndarray
    task : str — 'classification' or 'regression'
    
    Returns
    -------
    List of 5 rounds. Each round is a list of 2 folds.
    Each fold is a dict: {train_idx, test_idx}
    Total experiments per dataset: 10 (5 rounds × 2 folds)
    """
    rng = np.random.default_rng(random_state)
    n = len(y)
    rounds = []

    for _ in range(5):
        fold_a_idx = []
        fold_b_idx = []

        if task == 'classification':
            # Stratified: split each class in half
            classes = np.unique(y)
            for c in classes:
                class_idx = np.where(y == c)[0]
                shuffled = rng.permutation(class_idx)
                mid = len(shuffled) // 2
                fold_a_idx.extend(shuffled[:mid].tolist())
                fold_b_idx.extend(shuffled[mid:].tolist())

        elif task == 'regression':
            # Uniform: sort by target, alternate assignments
            sorted_idx = np.argsort(y)
            shuffled = rng.permutation(sorted_idx)  # shuffle within sorted order
            # Interleave: even indices → fold A, odd → fold B
            fold_a_idx = shuffled[0::2].tolist()
            fold_b_idx = shuffled[1::2].tolist()

        fold_a_idx = np.array(fold_a_idx)
        fold_b_idx = np.array(fold_b_idx)

        round_folds = [
            {"train_idx": fold_a_idx, "test_idx": fold_b_idx},
            {"train_idx": fold_b_idx, "test_idx": fold_a_idx},
        ]
        rounds.append(round_folds)

    return rounds  # shape: 5 rounds × 2 experiments

In [11]:
# Z-Score wired inside CV

def normalize_fold(X_train, X_test):
    """
    Fit z-score params on training fold only.
    Apply to both train and test.
    Test data plays no part in computing mu or sigma.
    
    Parameters
    ----------
    X_train : np.ndarray
    X_test  : np.ndarray
    
    Returns
    -------
    X_train_norm, X_test_norm, mu, sigma
    """
    mu, sigma = compute_z_score_normalization(X_train)
    X_train_norm = apply_zscore(X_train, mu, sigma)
    X_test_norm  = apply_zscore(X_test,  mu, sigma)
    return X_train_norm, X_test_norm, mu, sigma

In [12]:
# Null models

def null_classifier(y_train, X_test):
    """
    Null model for classification.
    Predicts the plurality (most common) class label for every test point.
    Does not look at any feature values.
    
    Parameters
    ----------
    y_train : np.ndarray — training labels
    X_test  : np.ndarray — test features (shape only used for output size)
    
    Returns
    -------
    np.ndarray — predicted labels, same length as X_test
    """
    classes, counts = np.unique(y_train, return_counts=True)
    plurality_class = classes[np.argmax(counts)]
    return np.full(len(X_test), plurality_class, dtype=y_train.dtype)


def null_regressor(y_train, X_test):
    """
    Null model for regression.
    Predicts the mean of training targets for every test point.
    Does not look at any feature values.
    
    Parameters
    ----------
    y_train : np.ndarray — training targets
    X_test  : np.ndarray — test features (shape only used for output size)
    
    Returns
    -------
    np.ndarray — predicted values, same length as X_test
    """
    mean_val = np.mean(y_train)
    return np.full(len(X_test), mean_val, dtype=float)

In [13]:
# Evaluation Metrics

def classification_error(y_true, y_pred):
    """1 - accuracy. Fraction of predictions that are wrong."""
    return np.mean(y_true != y_pred)


def mean_squared_error(y_true, y_pred):
    """Mean squared error for regression."""
    return np.mean((y_true - y_pred) ** 2)

In [14]:
# Scaffold Runner (Null models through full CV)

def run_cv(datasets, model_fn_clf, model_fn_reg, label="model"):
    """
    Run 5x2 CV for all datasets using provided model functions.
    
    model_fn_clf(X_train, y_train, X_test) -> y_pred  (classification)
    model_fn_reg(X_train, y_train, X_test) -> y_pred  (regression)
    
    Prints mean performance across 10 experiments per dataset.
    """
    print(f"\n{'='*55}")
    print(f"  Results: {label}")
    print(f"{'='*55}")

    for name, d in datasets.items():
        task   = d["task"]
        X_main = d["X_main"]
        y_main = d["y_main"]

        folds  = make_5x2_folds(X_main, y_main, task, random_state=RANDOM_STATE)
        scores = []

        for round_folds in folds:          # 5 rounds
            for fold in round_folds:       # 2 experiments per round
                tr_idx = fold["train_idx"]
                te_idx = fold["test_idx"]

                X_tr, X_te, _, _ = normalize_fold(X_main[tr_idx], X_main[te_idx])
                y_tr = y_main[tr_idx]
                y_te = y_main[te_idx]

                if task == "classification":
                    y_pred = model_fn_clf(X_tr, y_tr, X_te)
                    score  = classification_error(y_te, y_pred)
                else:
                    y_pred = model_fn_reg(X_tr, y_tr, X_te)
                    score  = mean_squared_error(y_te, y_pred)

                scores.append(score)

        metric = "class error" if task == "classification" else "MSE"
        print(f"  {name:20s} | {metric:11s} | mean: {np.mean(scores):.4f} | std: {np.std(scores):.4f}")


# Wrap null models to match the runner's expected signature
null_clf = lambda X_tr, y_tr, X_te: null_classifier(y_tr, X_te)
null_reg = lambda X_tr, y_tr, X_te: null_regressor(y_tr, X_te)

run_cv(datasets, null_clf, null_reg, label="Null Models")


  Results: Null Models
  breast_cancer        | class error | mean: 0.3453 | std: 0.0012
  congress_vote        | class error | mean: 0.3851 | std: 0.0000
  hw_performance       | MSE         | mean: 28538.5198 | std: 8608.3907
  abalone              | MSE         | mean: 10.4606 | std: 0.2046


In [15]:
# Minkowski Distance Function

def minkowski_distance(x, y, p=2):      # x & y are two data points, each is a 1D numpy array of feature values
                                        # p controls which version of distance we compute (2 = Euclidean)
                                        
    differences = np.abs(x - y)         # subtract each feature of y from the matching feature of x, np.abs takes the absolute value so negatives don't cancel positives. The result is an arracy of unsigned differences, one per feature. 
    
    powered = differences ** p          #  raise each difference to the power p, for p=2 this squares each difference, amplifying large gaps more than small ones.
    
    total = np.sum(powered)             # sum all the powered differences into a single number
    
    distance = total ** (1.0 / p)       # take the p-th root to bring the result back to the original unit scale, 1.0/p ensures float division and avoids integer rounding errors. 
    
    return distance

  

In [16]:
# VDM 

def fit_vdm(X_cat, y, p=2):             # X_cat is the catergorical feature matrix shape (n_samples, n_features). Y is the array of class labels, P is the norm order, the same P will be used later in the distance calculation
    
    classes = np.unique(y)              # find all unique class labels in the training set. For congressional vote this is [0, 1] (democract, republican)
    
    n_features = X_cat.shape[1]         # get the number of categorial feature columns
    
    vdm_tables = []                     # will hold one dictionary per feature, each dictionary maps a feature value to its class proportion array. 
    
    for f in range(n_features):         # loop over each ceature column one at a time
        feature_vals = np.unique(X_cat[:, f])   # find all unique values that appear in this feature column, for vote features this will be some subset of [0, 1, 2].
        table = {}                      # dictionary for this feature. Map value -> proportion array
        
        for v in feature_vals:          # loop over each unique value v in this feature
            mask = X_cat[:, f] == v     # boolean array, True whenever this feature equals value v
            
            C_i = np.sum(mask)          # count how many training examples have this feature value, this is the denominator in the proportion formula.
            
            proportions = np.array([            # for each class c, count the examples that have both value v and class, then divide by C_i to get the proportion. 
                np.sum(mask & (y==c)) / C_i     # result is one proportion per class, stored as an array. 
                for c in classes
            ])
            
            table[v] = proportions       # store the proportion array under this value in the table. 
            
        vdm_tables.append(table)        # add this feature's completed table to the list
        
    return {"tables": vdm_tables, "classes": classes, "p": p}       # return everything needed for the distance step bundled as a dictionary so we can pass it around as one object. 

#***    VDM Distance Function   ***#

def vdm_distance(x, y_vec, vdm_model):      # xa dn y_vec are two data points, each is a 1D array of categorical feature values, vdm_model is the dictionary returned by fit_vdm
    
    tables = vdm_model["tables"]        # pull out the list of per-feature proportion tables
    
    p = vdm_model["p"]                 # pull out the norm order used when the model was fit
    
    total = 0.0                         # accumulator for the total disctance across all features
    
    for f, table in enumerate(tables):      # loop over each feature, paired with its proportion table
        
        vi = x[f]       # value of feature f in point x
        
        vj = y_vec[f]   # value of feature f in point y_vec
        
        if vi == vj:        # if both points have the same value for this feature, the disctance contribution is zero, skip it.  
            continue
        
        pi = table.get(vi, np.ones(len(vdm_model["classes"])) / len(vdm_model["classes"]))  # look up the class proportion array for value vi, if vi was never seen in training then fall back to uniform proportions. uniform = equal probability for each class = maximally uncertain. 
        
        pj = table.get(vj, np.ones(len(vdm_model["classes"])) / len(vdm_model["classes"]))  # same for vj
        
        total += np.sum(np.abs(pi - pj) ** p)   # for each class, take the absolute difference in proportions, raise to the power p, sum across all classes, add to the running total, no p-th root, requirements say it dosen't change neighbor ordering
        
    return total        # total distance between x and y_vec across all categorical features

In [17]:
# Combined Distance

def combined_distance(x, y_vec, num_idx, cat_idx, vdm_model=None, p=2):     # x and y_vec are two full data points, 1D arrays with all features. num_idx is a list of which column positions are numeric features. cat_idx is a list of which column positions are categorical features. vdm_model is the fitted VDM table, only needed if cat_idx is non-empty. p is the norm order, same p used throughout
    
    dist = 0.0      # accumulator, will add numeric and categorical distances into this
    
    if len(num_idx) > 0:        # only run this block if there are numeric features to measure
        
        x_num = x[num_idx]      # pull out only the numeric feature values from point x
        
        y_num = y_vec[num_idx]      # pull out only the numeric feature values from point y_vec
        
        dist += np.sum(np.abs(x_num - y_num) ** p)      # compute Minkowski distance without the p-th root. skip it to keep combined distance addition clean. 
        
    if len(cat_idx) > 0 and vdm_model is not None:      # only run this block if there are categorical features and a fitted vdm model 
        
        x_cat = x[cat_idx]      # pull out only the categorical feature values from point x
        
        y_cat = y_vec[cat_idx]      # pull out only the categorical feature values from point y_vec
        
        dist += vdm_distance(x_cat, y_cat, vdm_model)       # compute VDM distance and add it to the running tool, VDM already returns a sum with no root, consistent with numeric side
        
    return dist         # total distance between x and y_vec across all feature types 

In [18]:
# k-NN Classifier

def knn_classify(X_train, y_train, X_test, k, num_idx, cat_idx, vdm_model=None, p=2):    # X_train is the full training feature matrix, y_train is the training labels, X_test is the set of points we want to predict.
                                                                                        # k is the number of neighbors to consult, num_idx is the list of numeric feature column positions. cat_idx is the list of categorical 
                                                                                        # positions. vdm_model is the fitted VDM table. p is the norm order for distance calculation. 
    predictions = []        # will collect one predicted label per test point
    
    for x_q in X_test:              # loop over each query point we need to classify
        distances = np.array([                                                  # compute distance from this query point to every training point, result is a 1D array of length n_train, one distance per training point. 
            combined_distance(x_q, x_tr, num_idx, cat_idx, vdm_model, p)
            for x_tr in X_train
        ])   
        
        nn_idx = np.argsort(distances)[:k]     # argsort returns the indices that would sort the distances array. Smallest distances come first, then the first k indices. There are the k nearest neighbors. 
         
        neighbor_labels = y_train[nn_idx]     # look up class labels of those k nearest neighbors
          
        classes, counts = np.unique(neighbor_labels, return_counts=True)        # count how many votes each class received among the k neighbors. Classes = array of unique class labels. counts = array of vote counts, matching position with classes. 
        
        max_count = np.max(counts)      # find the highest vote count
        
        candidates = classes[counts == max_count]       # find all classes that tied for the highest vote count usually just one class, but could be multiple on a tie. 
        
        predicted = candidates[np.random.randint(len(candidates))]      # pick one winner, if no tie this always picks the only candidate. If there is a tie, pick randomly among tied classes. 
        
        predictions.append(predicted)       # add the predicted label to the output list
        
    return np.array(predictions)        # return all predictions as a numpy array                                                                           
                                                                                        

In [19]:
# Feature Index Definitons

feature_meta = {
    
    "breast_cancer":{
        "num_idx": list(range(9)),
        "cat_idx": [],
        "vdm_model": None
    }, 
    
    "congress_vote": {
        "num_idx": [],
        "cat_idx": list(range(16)),
        "vdm_model": None
    },
    
    "hw_performance": {
        "num_idx": list(range(6)),
        "cat_idx": [],
        "vdm_model": None
        
    },
    
    "abalone": {
        "num_idx": list(range(10)),
        "cat_idx": [],
        "vdm_model": None
    } 
}

In [20]:
# K Tuning

def tune_k(dataset_name, d, meta, k_values, p=2):
    X_train, X_tune_norm, _, _ = normalize_fold(d["X_main"], d["X_tune"])
    y_train = d["y_main"]
    y_tune = d["y_tune"]
    vdm_model = None
    
    if len(meta["cat_idx"]) > 0:
        vdm_model = fit_vdm(X_train[:, meta["cat_idx"]], y_train, p=p)
        
    best_k = k_values[0]
    best_error = float("inf")
    print(f"\n  Tuning k — {dataset_name}")
    # header line so output is readable
    
    for k in k_values:
        # test each candidate k value one at a time
        
        y_pred = knn_classify(
            X_train, y_train, X_tune_norm, k,
            meta["num_idx"], meta["cat_idx"], vdm_model, p
        )
        # run k-NN classifier using X_main as training and X_tune as test
        # this is the one-time burn of the tuning set
        
        error = classification_error(y_tune, y_pred)
        # compute fraction of tuning points predicted incorrectly
        
        print(f"    k={k:3d} | error: {error:.4f}")
        # print result for this k so we can see the full picture
        
        if error < best_error:
            # this k performed better than anything seen so far
            
            best_error = error
            # update the best error
            
            best_k = k
            # update the best k
    
    print(f"  Best k: {best_k} (error: {best_error:.4f})")
    # report the winner
    
    return best_k
    # return the best k for use in the final CV experiments


# Run tuning on both classification datasets
# Spec requires at least 4 candidate values
k_candidates = [1, 3, 5, 7, 9, 11, 15]

best_k = {}
# dictionary to store the winning k for each classification dataset

for name, d in datasets.items():
    # loop over all datasets
    
    if d["task"] == "classification":
        # only tune k for classification datasets — regression uses kernel bandwidth
        
        best_k[name] = tune_k(name, d, feature_meta[name], k_candidates)
        # run tuning and store the result


  Tuning k — breast_cancer
    k=  1 | error: 0.0429
    k=  3 | error: 0.0500
    k=  5 | error: 0.0500
    k=  7 | error: 0.0571
    k=  9 | error: 0.0571
    k= 11 | error: 0.0500
    k= 15 | error: 0.0500
  Best k: 1 (error: 0.0429)

  Tuning k — congress_vote
    k=  1 | error: 0.0460
    k=  3 | error: 0.0345
    k=  5 | error: 0.0345
    k=  7 | error: 0.0345
    k=  9 | error: 0.0345
    k= 11 | error: 0.0345
    k= 15 | error: 0.0345
  Best k: 3 (error: 0.0345)


In [21]:
# k-NN Wired Into run_cv

def make_knn_clf(name, meta, k, p=2):
    # name is the dataset name string — used to look up feature meta
    # meta is the feature_meta entry for this dataset
    # k is the tuned k value selected in cell 19
    # p is the norm order for distance calculation
    
    def clf(X_tr, y_tr, X_te):
        # this inner function matches the signature run_cv expects
        # X_tr, y_tr are the training fold — X_te is the test fold
        
        vdm_model = None
        # start with no VDM — only build it if dataset has categorical features
        
        if len(meta["cat_idx"]) > 0:
            # this dataset has categorical features
            
            vdm_model = fit_vdm(X_tr[:, meta["cat_idx"]], y_tr, p=p)
            # fit VDM on the categorical columns of this training fold only
            # fitted fresh each fold so no test data leaks into the model
        
        return knn_classify(
            X_tr, y_tr, X_te, k,
            meta["num_idx"], meta["cat_idx"], vdm_model, p
        )
        # run k-NN with the tuned k and return predictions
    
    return clf
    # return the inner function — run_cv will call it on each fold


# Run 5x2 CV for each classification dataset using tuned k values
for name, d in datasets.items():
    # loop over all datasets
    
    if d["task"] != "classification":
        continue
        # skip regression datasets — those come on day 5
    
    k = best_k[name]
    # retrieve the tuned k value selected in cell 19
    
    clf_fn = make_knn_clf(name, feature_meta[name], k, p=2)
    # build the classifier function with tuned k locked in
    
    run_cv(
        {name: d},
        clf_fn,
        lambda X_tr, y_tr, X_te: null_regressor(y_tr, X_te),
        # regression function is unused here — passing null as placeholder
        label=f"k-NN (k={k}) — {name}"
    )


  Results: k-NN (k=1) — breast_cancer
  breast_cancer        | class error | mean: 0.0318 | std: 0.0100

  Results: k-NN (k=3) — congress_vote
  congress_vote        | class error | mean: 0.0575 | std: 0.0184


In [22]:
# Gaussian Kernal

def gaussian_kernel(distance, gamma):       # distance is the Euclidean distance between a query point and one neighbor. Gamma is the badnwidth parameter, controls how fast weight drops with distance. 
    
    exponent = -gamma * (distance ** 2)     # multiply negative gamma by the squared distance, squaring amplifies large distances, far neighbors get penalized more. Negative sign ensure the exponent is always <= 0. Output will always be between 0 and 1
    
    return np.exp(exponent)                 # raise e to the exponent, when distance = 0: exp(0) = 1.0 - maximum weight. When distance is large: exp(very negative) -> 0.0 - minimal weight



In [23]:
def knn_regress(X_train, y_train, X_test, k, gamma, num_idx, cat_idx, vdm_model=None, p=2):
    # added p=2 back to signature
    
    predictions = []
    
    for x_q in X_test:
        if len(cat_idx) == 0:
            # numeric-only: vectorized distance across all training points at once
            distances = np.sum(np.abs(X_train[:, num_idx] - x_q[num_idx]) ** p, axis=1)
        else:
            distances = np.array([
                combined_distance(x_q, x_tr, num_idx, cat_idx, vdm_model, p)
                for x_tr in X_train
            ])
        
        nn_idx = np.argsort(distances)[:k]
        
        neighbor_targets = y_train[nn_idx]
        # fixed name — removed the s, consistent throughout
        
        neighbor_distances = distances[nn_idx]
        # fixed name — removed the s, consistent throughout
        
        weights = np.array([
            gaussian_kernel(d, gamma)
            for d in neighbor_distances
        ])
        
        weight_sum = np.sum(weights)
        # this line was missing — must compute before using it below
        
        if weight_sum == 0:
            prediction = np.mean(neighbor_targets)
        else:
            prediction = np.sum(weights * neighbor_targets) / weight_sum
        
        predictions.append(prediction)
    
    return np.array(predictions)


In [24]:
# Tune k and Gamma for Regression

def tune_regression(dataset_name, d, meta, k_values, gamma_values, p=2):
    
    X_train, X_tune_norm, _, _ = normalize_fold(d["X_main"], d["X_tune"])
    
    y_train = d["y_main"]
    
    y_tune = d["y_tune"]
    
    best_k = k_values[0]
    
    best_gamma = gamma_values[0]
    
    best_mse = float("inf")
    
    print(f"\n Tuning k and gamma - {dataset_name}")
    
    for k in k_values: 
        for gamma in gamma_values:
            y_pred = knn_regress(
                X_train, y_train, X_tune_norm, k, gamma,
                meta["num_idx"], meta["cat_idx"], None, p
            )
            
            mse = mean_squared_error(y_tune, y_pred)
            
            print(f"k={k:3d} | gamma={gamma:.4f} | MSE: {mse:.4f}")
            
            if mse < best_mse:
                best_mse = mse
                best_k = k
                best_gamma = gamma
                
    print(f"\n  Best k: {best_k} | Best gamma: {best_gamma} | MSE: {best_mse:.4f}")
    # report the winning combination
    
    return best_k, best_gamma
    # return both tuned values for use in final CV experiments


# Candidate values — at least 4 per hyperparameter per spec
k_candidates_reg    = [1, 3, 5, 7]
# start conservative — regression datasets are smaller than classification

gamma_candidates    = [0.1, 0.5, 1.0, 5.0]
# span a wide range — we don't know the scale yet
# 0.1 = gentle dropoff, 5.0 = steep dropoff

best_k_reg     = {}
# store best k per regression dataset

best_gamma_reg = {}
# store best gamma per regression dataset

for name, d in datasets.items():
    # loop over all datasets
    
    if d["task"] != "regression":
        continue
        # skip classification datasets
    
    best_k_reg[name], best_gamma_reg[name] = tune_regression(
        name, d, feature_meta[name], k_candidates_reg, gamma_candidates
    )


 Tuning k and gamma - hw_performance
k=  1 | gamma=0.1000 | MSE: 2466.6190
k=  1 | gamma=0.5000 | MSE: 2466.6190
k=  1 | gamma=1.0000 | MSE: 2466.6190
k=  1 | gamma=5.0000 | MSE: 2466.6190
k=  3 | gamma=0.1000 | MSE: 1755.2463
k=  3 | gamma=0.5000 | MSE: 1668.9695
k=  3 | gamma=1.0000 | MSE: 1683.8646
k=  3 | gamma=5.0000 | MSE: 1927.1080
k=  5 | gamma=0.1000 | MSE: 1444.1949
k=  5 | gamma=0.5000 | MSE: 1491.5511
k=  5 | gamma=1.0000 | MSE: 1540.1072
k=  5 | gamma=5.0000 | MSE: 1870.5574
k=  7 | gamma=0.1000 | MSE: 1445.0793
k=  7 | gamma=0.5000 | MSE: 1549.2257
k=  7 | gamma=1.0000 | MSE: 1592.4592
k=  7 | gamma=5.0000 | MSE: 1881.1275

  Best k: 5 | Best gamma: 0.1 | MSE: 1444.1949

 Tuning k and gamma - abalone
k=  1 | gamma=0.1000 | MSE: 9.2635
k=  1 | gamma=0.5000 | MSE: 9.2635
k=  1 | gamma=1.0000 | MSE: 9.2635
k=  1 | gamma=5.0000 | MSE: 9.2635
k=  3 | gamma=0.1000 | MSE: 6.0424
k=  3 | gamma=0.5000 | MSE: 6.0583
k=  3 | gamma=1.0000 | MSE: 6.0725
k=  3 | gamma=5.0000 | MSE: 6.

In [25]:
# k-NN Regressor Wired into run_cv

def make_knn_reg(name, meta, k, gamma, p=2):
    def reg(X_tr, y_tr, X_te):
        return knn_regress(
            X_tr, y_tr, X_te, k, gamma,
            meta["num_idx"], meta["cat_idx"], None, p
        )
        
    return reg

for name, d in datasets.items():
    
    if d["task"] != "regression":
        continue
    
    k = best_k_reg[name]
    
    gamma = best_gamma_reg[name]
    
    reg_fn = make_knn_reg(name, feature_meta[name], k, gamma, p=2)
    
    run_cv(
        {name: d},
        lambda X_tr, y_tr, X_te: null_classifier(X_tr, X_te),
        reg_fn, 
        label=f"k-NN (k={k}, gamma={gamma}) - {name}"
    )


  Results: k-NN (k=5, gamma=0.1) - hw_performance
  hw_performance       | MSE         | mean: 6498.0984 | std: 4647.1006

  Results: k-NN (k=7, gamma=0.1) - abalone
  abalone              | MSE         | mean: 5.2919 | std: 0.1213


In [ ]:
# Condensed k-NN

def condensed_knn(X_train, y_train, task, num_idx, cat_idx, epsilon=None, p=2):
    
    condensed_indices = {0}
    # track which training indices are already in the condensed set
    # prevents any training point from being added more than once
    # without this, points with identical features but labels > epsilon apart
    # cause the condensed set to grow unboundedly (one extra copy per pass)
    
    condensed_X = [X_train[0]]
    condensed_y = [y_train[0]]
    changed = True
    
    while changed:
        changed = False
        
        c_X = np.array(condensed_X)
        # convert to numpy array ONCE per pass — not inside the for loop
        
        c_y = np.array(condensed_y)
        # same — once per pass only
        
        for i in range(1, len(X_train)):
            
            if i in condensed_indices:
                continue
            # skip points already in the condensed set — they can't help further
            
            x_i = X_train[i]
            y_i = y_train[i]
            
            if len(cat_idx) == 0:
                # numeric-only: vectorized distance across all condensed points at once
                distances = np.sum(np.abs(c_X[:, num_idx] - x_i[num_idx]) ** p, axis=1)
            else:
                distances = np.array([
                    combined_distance(x_i, x_c, num_idx, cat_idx, None, p)
                    for x_c in c_X
                ])
            
            nn_idx = np.argsort(distances)[0]
            nearest_label = c_y[nn_idx]
            
            if task == "classification":
                if nearest_label != y_i:
                    condensed_X.append(x_i)
                    condensed_y.append(y_i)
                    condensed_indices.add(i)
                    changed = True
                    
            elif task == "regression":
                if abs(nearest_label - y_i) > epsilon:
                    condensed_X.append(x_i)
                    condensed_y.append(y_i)
                    condensed_indices.add(i)
                    changed = True
    
    return np.array(condensed_X), np.array(condensed_y)


In [27]:
# Tune Epsilon for Regression

def tune_epsilon(dataset_name, d, meta, epsilon_values, k, gamma, p=2):
    
    X_train, X_tune_norm, _, _ = normalize_fold(d["X_main"], d["X_tune"])
    
    y_train = d["y_main"]
    
    y_tune = d["y_tune"]
    
    target_range = np.max(y_train) - np.min(y_train)
    
    print(f"\n Tuning epsilon - {dataset_name}")
    print(f" Target range: {target_range:.2f} | min: {np.min(y_train):.2f} | max: {np.max(y_train):.2f} ")
    
    best_epsilon = epsilon_values[0]
    
    best_mse = float("inf")
    
    for epsilon in epsilon_values:
        
        c_X, c_y = condensed_knn(
            X_train, y_train, "regression",
            meta["num_idx"], meta["cat_idx"], epsilon=epsilon, p=p
        )
        
        compression = len(c_y) / len(y_train) * 100
        
        y_pred = knn_regress(
            c_X, c_y, X_tune_norm, k, gamma, 
            meta["num_idx"], meta["cat_idx"], None, p
        )
        
        mse = mean_squared_error(y_tune, y_pred)
        
        print(f"    epsilon={epsilon:.2f} | condensed size: {len(c_y):4d}/{len(y_train)} ({compression:.1f}%) | MSE: {mse:.4f}")
        # print result for this epsilon — size and MSE together tell the full story
        
        if mse < best_mse:
            # this epsilon performed better than anything seen so far
            
            best_mse     = mse
            # update best MSE
            
            best_epsilon = epsilon
            # update best epsilon
    
    print(f"\n  Best epsilon: {best_epsilon} | MSE: {best_mse:.4f}")
    # report the winner
    
    return best_epsilon
    # return tuned epsilon for use in final CV experiments


# Candidate epsilon values — based on target ranges
# Hardware PRP range is roughly 6 to 1150 — test fractions of that range
# Abalone rings range is 1 to 29 — test fractions of that range
epsilon_candidates = {
    "hw_performance": [10, 25, 50, 100],
    # fractions of the ~1144 PRP range — from tight to loose tolerance
    
    "abalone": [0.5, 1.0, 2.0, 3.0]
    # fractions of the 28-ring range — from tight to loose tolerance
}

best_epsilon = {}
# store best epsilon per regression dataset

for name, d in datasets.items():
    # loop over all datasets
    
    if d["task"] != "regression":
        continue
        # skip classification datasets
    
    best_epsilon[name] = tune_epsilon(
        name, d, feature_meta[name],
        epsilon_candidates[name],
        best_k_reg[name],
        best_gamma_reg[name]
    )
    # run tuning and store result


 Tuning epsilon - hw_performance
 Target range: 1144.00 | min: 6.00 | max: 1150.00 


KeyboardInterrupt: 